# Fixture Enrichment

Validate the World Cup 2026 fixture and team reference inputs before joining fixtures with team metadata and Elo ratings.

In [1]:
from pathlib import Path

import pandas as pd

## Paths

In [2]:
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

FIXTURES_RAW_PATH = DATA_RAW / "FIFA_WC_1930_2026" / "wc_2026_fixtures.csv"
TEAMS_RAW_PATH = DATA_RAW / "FIFA_WC_1930_2026" / "wc_2026_teams.csv"
ELO_LATEST_PATH = DATA_PROCESSED / "elo_latest.csv"

FIXTURES_VALIDATED_PATH = DATA_PROCESSED / "wc_2026_fixtures_validated.csv"
TEAMS_CLEANED_PATH = DATA_PROCESSED / "wc_2026_teams_cleaned.csv"

## Load inputs

In [3]:
fixtures = pd.read_csv(FIXTURES_RAW_PATH)
teams = pd.read_csv(TEAMS_RAW_PATH)
elo_latest = pd.read_csv(ELO_LATEST_PATH)

print("fixtures:", fixtures.shape)
print("teams:", teams.shape)
print("elo_latest:", elo_latest.shape)

fixtures: (104, 15)
teams: (48, 7)
elo_latest: (48, 23)


## Parse dates

In [4]:
fixtures["date"] = pd.to_datetime(
    fixtures["date"],
    format="%Y-%m-%d",
    errors="raise",
)

print("min date:", fixtures["date"].min())
print("max date:", fixtures["date"].max())

min date: 2026-06-11 00:00:00
max date: 2026-07-19 00:00:00


## Identify known teams and placeholders

In [5]:
qualified_teams = set(teams["team"])
elo_teams = set(elo_latest["country"])
fixture_teams = set(fixtures["team1"]) | set(fixtures["team2"])

old_team_names = {"Turkey", "United States", "Czech Republic"}
canonical_team_names = {"Türkiye", "USA", "Czechia"}

fixtures["team1_is_known"] = fixtures["team1"].isin(qualified_teams)
fixtures["team2_is_known"] = fixtures["team2"].isin(qualified_teams)
fixtures["is_placeholder_match"] = ~(
    fixtures["team1_is_known"] & fixtures["team2_is_known"]
)

known_fixture_teams = (
    set(fixtures.loc[fixtures["team1_is_known"], "team1"])
    | set(fixtures.loc[fixtures["team2_is_known"], "team2"])
)

fixtures.groupby("stage")["is_placeholder_match"].agg(["sum", "count"])

,sum,count
stage,,
3rd Place Match,1,1
Final,1,1
Group Stage,0,72
Quarter-final,4,4
Round of 16,8,8
Round of 32,16,16
Semi-final,2,2


## Validate team compatibility

In [6]:
assert len(teams) == 48
assert teams["team"].nunique() == 48
assert teams.isna().sum().sum() == 0
assert teams.duplicated("team").sum() == 0

assert set(teams["team"]) == elo_teams
assert known_fixture_teams == qualified_teams
assert known_fixture_teams <= elo_teams

assert old_team_names.isdisjoint(fixture_teams | qualified_teams | elo_teams)
assert canonical_team_names.issubset(fixture_teams)
assert canonical_team_names.issubset(qualified_teams)
assert canonical_team_names.issubset(elo_teams)

print("Team compatibility checks passed.")

Team compatibility checks passed.


## Validate fixture structure

In [7]:
expected_stage_counts = {
    "Group Stage": 72,
    "Round of 32": 16,
    "Round of 16": 8,
    "Quarter-final": 4,
    "Semi-final": 2,
    "3rd Place Match": 1,
    "Final": 1,
}

match_pair_key = fixtures.apply(
    lambda row: tuple(sorted([row["team1"], row["team2"]])),
    axis=1,
)

assert len(fixtures) == 104
assert fixtures["stage"].value_counts().to_dict() == expected_stage_counts
assert fixtures["date"].notna().all()
assert fixtures["date"].min() == pd.Timestamp("2026-06-11")
assert fixtures["date"].max() == pd.Timestamp("2026-07-19")

assert fixtures.duplicated().sum() == 0
assert fixtures.duplicated(["stage", "date", "team1", "team2"]).sum() == 0
assert fixtures.duplicated(["date", "team1", "team2"]).sum() == 0
assert (
    fixtures.assign(match_pair_key=match_pair_key)
    .duplicated(["stage", "date", "match_pair_key"])
    .sum()
    == 0
)

print("Fixture structure checks passed.")

Fixture structure checks passed.


## Validate placeholders and expected missing values

In [8]:
metadata_columns = [
    "team1_confederation",
    "team1_fifa_rank",
    "team1_coach",
    "team2_confederation",
    "team2_fifa_rank",
    "team2_coach",
]

known_matches = ~fixtures["is_placeholder_match"]
placeholder_matches = fixtures["is_placeholder_match"]

assert known_matches.sum() == 72
assert placeholder_matches.sum() == 32
assert fixtures.loc[known_matches, "stage"].eq("Group Stage").all()
assert fixtures.loc[placeholder_matches, "stage"].ne("Group Stage").all()
assert fixtures.loc[known_matches, metadata_columns].notna().all().all()
assert fixtures.loc[placeholder_matches, metadata_columns].isna().all().all()
assert fixtures.loc[known_matches, "group"].notna().all()
assert fixtures.loc[placeholder_matches, "group"].isna().all()

print("Placeholder and expected missing-value checks passed.")

Placeholder and expected missing-value checks passed.


## Save validated handoff files

In [9]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

fixtures.to_csv(FIXTURES_VALIDATED_PATH, index=False)
teams.to_csv(TEAMS_CLEANED_PATH, index=False)

print(f"Saved {FIXTURES_VALIDATED_PATH}")
print(f"Saved {TEAMS_CLEANED_PATH}")

Saved ../data/processed/wc_2026_fixtures_validated.csv
Saved ../data/processed/wc_2026_teams_cleaned.csv


## Load enrichment inputs

In [10]:
fixtures = pd.read_csv(FIXTURES_VALIDATED_PATH, parse_dates=["date"])
teams = pd.read_csv(TEAMS_CLEANED_PATH)
elo_latest = pd.read_csv(ELO_LATEST_PATH)

fixtures.head()

,group,stage,team1,team2,venue,city,country,date,kickoff_et,team1_confederation,team1_fifa_rank,team1_coach,team2_confederation,team2_fifa_rank,team2_coach,team1_is_known,team2_is_known,is_placeholder_match
0,A,Group Stage,Mexico,South Africa,Estadio Azteca,Mexico City,Mexico,2026-06-11,20:00 ET,CONCACAF,15.0,Javier Aguirre,CAF,60.0,Hugo Broos,True,True,False
1,A,Group Stage,South Korea,Czechia,Estadio Akron,Guadalajara,Mexico,2026-06-11,22:00 ET,AFC,25.0,Hong Myung-bo,UEFA,41.0,Ivan Hasek,True,True,False
2,A,Group Stage,South Korea,Mexico,Estadio Akron,Guadalajara,Mexico,2026-06-18,21:00 ET,AFC,25.0,Hong Myung-bo,CONCACAF,15.0,Javier Aguirre,True,True,False
3,A,Group Stage,Czechia,South Africa,Estadio Akron,Guadalajara,Mexico,2026-06-18,18:00 ET,UEFA,41.0,Ivan Hasek,CAF,60.0,Hugo Broos,True,True,False
4,A,Group Stage,Czechia,Mexico,Estadio Azteca,Mexico City,Mexico,2026-06-24,21:00 ET,UEFA,41.0,Ivan Hasek,CONCACAF,15.0,Javier Aguirre,True,True,False


In [11]:
print(fixtures[["team1", "team2", "is_placeholder_match"]].head())
print(elo_latest.columns)

         team1         team2  is_placeholder_match
0       Mexico  South Africa                 False
1  South Korea       Czechia                 False
2  South Korea        Mexico                 False
3      Czechia  South Africa                 False
4      Czechia        Mexico                 False
Index(['year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating',
       'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min',
       'rating_min', 'matches_total', 'matches_home', 'matches_away',
       'matches_neutral', 'wins', 'losses', 'draws', 'goals_for',
       'goals_against', 'confederation', 'is_host'],
      dtype='str')


In [ ]:
known_fixtures = fixtures[fixtures["is_placeholder_match"] == False].copy()

team1_elo = elo_latest.add_prefix("team1_elo_")

known_fixtures = known_fixtures.merge(
    team1_elo, 
    left_on="team1",
    right_on="team1_elo_country",
    how="left"
)


print(known_fixtures[["team1", "team2", "team1_elo_country", "team1_elo_rating"]].head())

         team1         team2 team1_elo_country  team1_elo_rating
0       Mexico  South Africa            Mexico              1860
1  South Korea       Czechia       South Korea              1752
2  South Korea        Mexico       South Korea              1752
3      Czechia  South Africa           Czechia              1726
4      Czechia        Mexico           Czechia              1726
0


In [17]:
known_fixtures["team1_elo_rating"].isna().sum()

np.int64(0)

In [18]:
team2_elo = elo_latest.add_prefix("team2_elo_")

known_fixtures = known_fixtures.merge(
    team2_elo,
    left_on="team2",
    right_on="team2_elo_country", 
    how="left"
)

known_fixtures[[
    "team1", "team2",
    "team1_elo_rating", "team2_elo_rating"
]].head()

,team1,team2,team1_elo_rating,team2_elo_rating
0,Mexico,South Africa,1860,1524
1,South Korea,Czechia,1752,1726
2,South Korea,Mexico,1752,1860
3,Czechia,South Africa,1726,1524
4,Czechia,Mexico,1726,1860


In [19]:
known_fixtures[["team1_elo_rating", "team2_elo_rating"]].isna().sum()

team1_elo_rating    0
team2_elo_rating    0
dtype: int64

In [24]:
known_fixtures.columns

Index(['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date',
       'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach',
       'team2_confederation', 'team2_fifa_rank', 'team2_coach',
       'team1_is_known', 'team2_is_known', 'is_placeholder_match',
       'team1_elo_year', 'team1_elo_snapshot_date', 'team1_elo_country',
       'team1_elo_rank', 'team1_elo_country_code', 'team1_elo_rating',
       'team1_elo_rank_max', 'team1_elo_rating_max', 'team1_elo_rank_avg',
       'team1_elo_rating_avg', 'team1_elo_rank_min', 'team1_elo_rating_min',
       'team1_elo_matches_total', 'team1_elo_matches_home',
       'team1_elo_matches_away', 'team1_elo_matches_neutral', 'team1_elo_wins',
       'team1_elo_losses', 'team1_elo_draws', 'team1_elo_goals_for',
       'team1_elo_goals_against', 'team1_elo_confederation',
       'team1_elo_is_host', 'team2_elo_year', 'team2_elo_snapshot_date',
       'team2_elo_country', 'team2_elo_rank', 'team2_elo_country_code',
 

In [25]:
known_fixtures.shape

(72, 65)

In [23]:
known_fixtures["elo_diff"] = (
    known_fixtures["team1_elo_rating"] 
    - known_fixtures["team2_elo_rating"]
)

known_fixtures["elo_diff"]

0     336
1      26
2    -108
3     202
4    -134
     ... 
67   -234
68    517
69    193
70   -283
71    427
Name: elo_diff, Length: 72, dtype: int64

In [26]:
known_fixtures.shape

(72, 65)

In [27]:
known_fixtures[
    [
        "team1_elo_rating",
        "team2_elo_rating"
    ]
].isna().sum()

team1_elo_rating    0
team2_elo_rating    0
dtype: int64